<a href="https://colab.research.google.com/github/women-in-ai-ireland/June-2024-Group-002/blob/embeddings/Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embeddings

Embeddings using [Instructor Embedding](https://huggingface.co/hkunlp)

#### Prepare environment

In [23]:
# install libraries
!pip install langchain -U langchain-community faiss-cpu langchain-openai tiktoken unstructured selenium newspaper3k textstat

!pip install sentence-transformers==2.2.2
!pip install InstructorEmbedding

!pip install pypdf

In [68]:
# import libraries

import pickle
import os #For OpenAI API
import getpass
from langchain.document_loaders import WebBaseLoader, UnstructuredURLLoader, NewsURLLoader, SeleniumURLLoader

from langchain_openai import OpenAI # This is to interact with OpenAI
from langchain_community.document_loaders import TextLoader # To load text file
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings #To convert our text to embedings
from langchain_text_splitters import RecursiveCharacterTextSplitter # To split text to reduce token size
from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import PyPDFLoader

# InstructorEmbedding
from InstructorEmbedding import INSTRUCTOR
from langchain.embeddings import HuggingFaceInstructEmbeddings

from langchain.chains import RetrievalQA

from pypdf import PdfReader

In [4]:
# access HF secret
from google.colab import userdata
userdata.get('HF_TOKEN')

In [5]:
# set open ai api key
os.environ["OPENAI_API_KEY"] = "OPEN AI KEY"

In [64]:
# connect to Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
root_dir = "/content/gdrive/MyDrive/WAI_project/"

Mounted at /content/gdrive


In [ ]:
# load instructor embeddings model
instructor_embeddings = HuggingFaceInstructEmbeddings(model_name="hkunlp/instructor-xl",
                                                      model_kwargs={"device": "cuda"})

In [65]:
# set local path to store embeddings ***replace with SingleStore, AWS or similar
embedding_store_path = f"{root_dir}/embedding_store"

#### Helper functions

In [15]:
# function that loads a web document into a LangChain document object

def load_document_from_url(loader_class, website_url):
    """
    Load a document using the specified loader class and website URL.

    Args:
    loader_class (class): The class of the loader to be used  i.e. WebBaseLoader, SeleniumURLLoader, NewsURLLoader
    website_url (str): The URL of the website from which to load the document.

    Returns:
    str: The loaded document.
    """
    loader = loader_class([website_url])
    return loader.load()

In [ ]:
# function to store the embeddings locally
def store_embeddings(docs, embeddings, sotre_name, path):
    vectorStore = FAISS.from_documents(docs, embeddings)
    with open(f"{path}/faiss_{sotre_name}.pkl", "wb") as f:
      pickle.dump(vectorStore, f)

In [ ]:
# function to load the saved embeddings
def load_embeddings(sotre_name, path):
  with open(f"{path}/faiss_{sotre_name}.pkl", "rb") as f:
      VectorStore = pickle.load(f)
  return VectorStore

#### Read File

In [43]:
# loader = TextLoader('single_text_file.txt')
#loader = DirectoryLoader(f'/content/gdrive/MyDrive/WAI_project/', glob="./*.pdf", loader_cls=PyPDFLoader)
#documents = loader.load()

In [47]:
# read single pdf file (replace with DirectoryLoader as above)
file_path = ('/content/gdrive/MyDrive/WAI_project/AQA-8035-TG-NEE-CS-STUDENT-BOOKLET.PDF')
loader = PyPDFLoader(file_path)
documents = loader.load_and_split()

#### Get Embeddings from the file

In [ ]:
# test FAISS using Instructor Embeddings without saving into a file
db_instructEmbedd = FAISS.from_documents(documents, instructor_embeddings)
retriever = db_instructEmbedd.as_retriever(search_kwargs={"k": 3}) # return the 3 closests vectors
docs = retriever.invoke("Who is Fela Kuti?")
docs[0]

In [69]:
# store embeddings in a local folder
store_embeddings(docs,
                 instructor_embeddings,
                 sotre_name='instructEmbeddings',
                 path=embedding_store_path)

In [71]:
# load the embeddings
db_instructEmbedd = load_embeddings(sotre_name='instructEmbeddings',
                                    path=embedding_store_path)

In [72]:
# get embeddings from local db
retriever = db_instructEmbedd.as_retriever(search_kwargs={"k": 3})
docs = retriever.get_relevant_documents("Who is Fela Kuti?")
docs[0]

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


Document(metadata={'source': '/content/gdrive/MyDrive/WAI_project/AQA-8035-TG-NEE-CS-STUDENT-BOOKLET.PDF', 'page': 6}, page_content='GCSE GEOGRAPHY – 8035 – LIC/NEE CASE STUDY STUDENT WORKBOOK  \n© 2023 AQA  7 of 33 \n Figure 6. A map to show the largest religion in each province of Nigeria .  \n \n \n \n 3. What does Figure 6  tell you \nabout religion  across Nigeria ? \n \nFigure 7. The legacy of Fela Kuti, Nigerian musician and activist . \n \n \n \nFela Kuti was a Nigerian musician and activist . He \ncreated  a modern style of music called Afro -beat . Afro -\nbeat  combine s traditional Yoruba music with American \nblues, jazz and funk. Kuti often included political \nstatements about the Nigerian government  in his music \nto create social change . 4. What does Figure 7  tell you \nabout Nigeria’s entertainment \nindustry?')